# COMPARISON — Temporal-First Graph Network (TFGN) ablation ladder

Implements `DOCS/temporal-first-ablation.md`'s 2026-08-24 "Evaluation & Comparison
Protocol" addendum over the ladder in `DOCS/flipped/PLAN.md` Phase 4 /
`CLASSIFIER/experiments/temporal_first.yaml`. Every section through Tier 3 reads only
`run_summary["oof"]` / `oof_predictions.csv` — **never** `test_*` / `ext_*` keys. The
one frozen in-domain + one frozen OASIS-3 read live in the final section only, gated
behind an explicit flag (`RUN_FROZEN_READ`), so re-running this notebook does not
accidentally spend the ladder's one test read before it is meant to.

`source_experiment`-style: no training here — every number comes from
`outputs/<rung>-seed{42..45}/latest/run_summary.json` (`adapters.explain.resolve_source_run`),
written by `LONGITUDINAL_COMMON_DELCODE.ipynb`'s runs.

In [ ]:
# === Papermill parameters ===
EXPERIMENT_ID = None
MODE = None
MODEL = None
SEED = 42
WANDB_ENABLED = False
OUTPUT_DIR = None
RUN_DIR = None
RUN_NAME = None
# Tier-4 is gated: only flip this (and set FROZEN_WINNER_ID) once the ladder is
# frozen and you mean to spend the one in-domain + one OASIS-3 read.
RUN_FROZEN_READ = False
FROZEN_WINNER_ID = None       # e.g. 'tfgn-s1c-recon-pooled' (id prefix, seed appended)


## Pipeline overview

Resolve each rung's 4 seed runs -> Tier 1 floors -> Tier 2 rung table + stopping rule (paired seed-level OOF ΔAUC) -> Tier 3 vetoes -> Tier 4 frozen reads (gated).

In [ ]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))


In [ ]:
# reproducibility seeding -- must run before datasets / models.
from SHARED.seeding import set_seed, make_rng, make_torch_generator
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)


In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from adapters.explain import resolve_source_run
from common.comparison import paired_bootstrap_ci

warnings.filterwarnings('ignore')
print('Imports OK')


## Configuration — rung registry

One entry per ladder rung; extend this dict as S1c-SENS land (`DOCS/temporal-first-ablation.md` "The arms") -- nothing else in this notebook needs to change. `S0_demo` is the new Tier-1 demographics floor (`tfgn-s0-demo-pooled`); it has no runs yet until dispatched.

In [ ]:
SEEDS = [42, 43, 44, 45]

# name -> registry id prefix (seed appended as '-seed{42,43,44,45}').
RUNG_PREFIXES = {
    'S0a_logreg_drift':   'tfgn-s0-logreg-drift-pooled',
    'S0_demo':            'tfgn-s0-demo-pooled',
    'S0b_gelstm_frozen':  'tfgn-s0-gelstm-frozen-pooled',
    'S0c_gelstm_random':  'tfgn-s0-gelstm-random-pooled',
    'S0d_braintokengt':   'tfgn-s0-braintokengt-pooled',
    'S1_flip':            'tfgn-s1-flip-pooled',
    'S1b_ssl':             'tfgn-s1b-ssl-pooled',
    # 'S1c_recon': 'tfgn-s1c-recon-pooled',   # add once launched
    # 'S2_gate': ..., 'S3_fusion': ..., 'S4_attnpool': ..., 'S5_dualscore': ..., 'SENS': ...,
}

# The pre-registered chain (DOCS/temporal-first-ablation.md "The arms") -- each
# rung's stopping-rule comparison is against the rung immediately before it here,
# not against every other rung.
RUNG_CHAIN = ['S0c_gelstm_random', 'S1_flip', 'S1b_ssl']  # extend: ..., 'S1c_recon', 'S2_gate', ...
HEADLINE_CONTRASTS = {
    'S0c_vs_S1': ('S0c_gelstm_random', 'S1_flip'),
    # 'S0b_vs_S1c': ('S0b_gelstm_frozen', 'S1c_recon'),   # add once S1c exists
}


In [ ]:
def resolve_rung_runs(prefix):
    """exp_id -> run_dir for every seed of one rung; missing runs are skipped, not fatal
    (this notebook must stay runnable before every rung/seed has been dispatched)."""
    run_dirs = {}
    for seed in SEEDS:
        exp_id = f'{prefix}-seed{seed}'
        try:
            run_dirs[exp_id] = resolve_source_run(exp_id, classifier_root=model_root)
        except FileNotFoundError:
            print(f'  [skip] {exp_id}: no run yet')
    return run_dirs


RUNG_RUN_DIRS = {name: resolve_rung_runs(prefix) for name, prefix in RUNG_PREFIXES.items()}
for name, dirs in RUNG_RUN_DIRS.items():
    print(f'{name:22s} {len(dirs)}/{len(SEEDS)} seeds resolved')


In [ ]:
def load_summary(run_dir):
    """None (not a raised error) if the run hasn't written run_summary.json yet --
    a run still 'running' on the other host is a normal state this notebook must
    tolerate, not treat as missing/broken."""
    path = run_dir / 'run_summary.json'
    return json.loads(path.read_text()) if path.is_file() else None


def load_calibration(run_dir):
    path = run_dir / 'calibration.json'
    return json.loads(path.read_text()) if path.is_file() else {}


def rung_oof_rows(run_dirs):
    """One row per seed with that seed's oof.* metrics (empty 'oof' -> pre-addendum,
    not-yet-re-run artifact -- flagged, not silently dropped)."""
    rows = []
    for exp_id, run_dir in run_dirs.items():
        summary = load_summary(run_dir)
        if summary is None:
            print(f'  [in-flight] {exp_id}: no run_summary.json yet (run still running?) -- skipped.')
            continue
        oof = summary.get('oof')
        if not oof:
            print(f'  [stale] {exp_id}: no run_summary["oof"] -- re-run under the '
                  '2026-08-24 addendum contract before trusting this rung\'s table row.')
            continue
        cal = load_calibration(run_dir)
        row = {'exp_id': exp_id, 'seed': int(exp_id.rsplit('seed', 1)[-1]),
               'cohort_probe_auc': summary.get('cohort_probe_auc'),
               'ece_oof_cal': cal.get('ece_oof_cal')}
        row.update(oof)
        rows.append(row)
    return pd.DataFrame(rows)


## Tier 1 — floor gates

In [ ]:
# Demographics floor: tfgn-s0-demo-pooled (feature_set='demo', [age, sex] only).
DEMO_FLOOR = rung_oof_rows(RUNG_RUN_DIRS.get('S0_demo', {}))
if not DEMO_FLOOR.empty:
    print('Demographics floor (age+sex only), OOF AUC:',
          f"{DEMO_FLOOR['oof_auc'].mean():.4f} +/- {DEMO_FLOOR['oof_auc'].std():.4f}")
else:
    print('Demographics floor: no runs yet (dispatch tfgn-s0-demo-pooled-seed{42..45}).')

# SSL persistence baseline -- already computed by LONGITUDINAL_TFGN_SSL_POOLED.ipynb
# itself; nothing to compute here, just surface it.
_p2_dir = resolve_source_run('tfgn-nodelstm-ssl-pooled', classifier_root=model_root)
_p2_summary = load_summary(_p2_dir) or {}
PERSISTENCE_BASELINE = _p2_summary.get('persistence_baseline', {})
print('P2 SSL persistence baseline:', PERSISTENCE_BASELINE)


## Tier 2 — rung table (OOF only) + stopping rule

In [ ]:
RUNG_TABLES = {name: rung_oof_rows(dirs) for name, dirs in RUNG_RUN_DIRS.items()}

_cols = ['oof_auc', 'oof_pr_auc', 'oof_balanced_accuracy', 'oof_static_n1_auc', 'cohort_probe_auc']
summary_rows = []
for name, df in RUNG_TABLES.items():
    if df.empty:
        summary_rows.append({'rung': name, 'n_seeds': 0})
        continue
    row = {'rung': name, 'n_seeds': len(df)}
    for c in _cols:
        if c in df.columns:
            row[f'{c}_mean'] = df[c].mean()
            row[f'{c}_sd'] = df[c].std()
    cohort_cols = [c for c in df.columns if c.startswith('oof_auc_') and c != 'oof_auc']
    for c in cohort_cols:
        row[f'{c}_mean'] = df[c].mean()
    summary_rows.append(row)

RUNG_SUMMARY_TABLE = pd.DataFrame(summary_rows).set_index('rung')
RUNG_SUMMARY_TABLE


In [ ]:
def per_fold_auc(run_dir):
    """fold -> OOF AUC, from oof_predictions.csv. The StratifiedGroupKFold split in
    common.crossval.run_kfold_cv takes no seed/shuffle, so fold i is the SAME subject
    group across every seed and every rung of one dataset -- this is what makes
    fold-matched pairing across arms/seeds valid."""
    path = run_dir / 'oof_predictions.csv'
    if not path.is_file():
        return {}
    df = pd.read_csv(path)
    out = {}
    for fold, sub in df.groupby('fold'):
        if sub['label'].nunique() > 1:
            out[int(fold)] = roc_auc_score(sub['label'], sub['prob'])
    return out


def stopping_rule(rung_k_dirs, rung_km1_dirs):
    """mean(Delta) / SE(Delta) of the seed-level mean paired fold-matched OOF ΔAUC
    (rung k vs rung k-1) -- DOCS/temporal-first-ablation.md 'The stopping rule'
    (2026-08-24 addendum: OOF, not in-domain test)."""
    seed_means = []
    for exp_id_k, dir_k in rung_k_dirs.items():
        seed = exp_id_k.rsplit('seed', 1)[-1]
        matches = [d for eid, d in rung_km1_dirs.items() if eid.endswith(f'seed{seed}')]
        if not matches:
            continue
        auc_k, auc_km1 = per_fold_auc(dir_k), per_fold_auc(matches[0])
        common_folds = sorted(set(auc_k) & set(auc_km1))
        if not common_folds:
            continue
        seed_means.append(float(np.mean([auc_k[f] - auc_km1[f] for f in common_folds])))
    if len(seed_means) < 2:
        return {'mean': float('nan'), 'se': float('nan'), 'ratio': float('nan'),
                'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': None}
    mean = float(np.mean(seed_means))
    se = float(np.std(seed_means, ddof=1) / np.sqrt(len(seed_means)))
    return {'mean': mean, 'se': se, 'ratio': (mean / se) if se > 0 else float('nan'),
            'n_seeds': len(seed_means), 'seed_means': seed_means, 'kept': mean > se}


print('Chain-adjacent stopping-rule decisions (OOF, per-seed paired fold-matched ΔAUC):')
for k in range(1, len(RUNG_CHAIN)):
    a, b = RUNG_CHAIN[k - 1], RUNG_CHAIN[k]
    result = stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    verdict = ('worth carrying forward' if result['kept'] else
               'undetectable at this sample size' if result['kept'] is not None else
               'not enough seeds resolved yet')
    print(f"  {b} vs {a}: mean(D)={result['mean']:.5f} SE(D)={result['se']:.5f} "
          f"ratio={result['ratio']:.2f} n_seeds={result['n_seeds']} -> {verdict}")

print()
print('Headline contrasts (isolate the flip itself):')
for label, (a, b) in HEADLINE_CONTRASTS.items():
    result = stopping_rule(RUNG_RUN_DIRS.get(b, {}), RUNG_RUN_DIRS.get(a, {}))
    print(f'  {label}: {result}')


## Tier 3 — robustness vetoes

Thresholds are fixed in `DOCS/temporal-first-ablation.md`'s addendum and never adjusted after seeing a result.

In [ ]:
VETO_THRESHOLDS = {
    'threshold_sd': 0.15,      # SD of best_threshold across 5 folds x 4 seeds
    'ece_oof_cal': 0.10,       # temperature-scaled OOF ECE
    'scan_count_spearman': 0.3,  # |r| of prob vs n_scans, within-stable
}


def veto_row(name, run_dirs):
    df = rung_oof_rows(run_dirs)
    if df.empty:
        return {'rung': name, 'n_seeds': 0}
    thresholds = []
    for run_dir in run_dirs.values():
        summary = load_summary(run_dir)
        if summary is None:
            continue
        thresholds.extend(summary.get('cv_results', {}).get('best_threshold', []))
    threshold_sd = float(np.std(thresholds, ddof=1)) if len(thresholds) > 1 else float('nan')

    row = {
        'rung': name,
        'threshold_sd': threshold_sd,
        'threshold_sd_veto': threshold_sd > VETO_THRESHOLDS['threshold_sd'],
        'ece_oof_cal': df['ece_oof_cal'].mean() if 'ece_oof_cal' in df else float('nan'),
    }
    row['ece_veto'] = (row['ece_oof_cal'] > VETO_THRESHOLDS['ece_oof_cal']
                        if pd.notna(row['ece_oof_cal']) else None)
    if 'oof_prob_nscans_spearman_non_converter' in df.columns:
        r = df['oof_prob_nscans_spearman_non_converter'].mean()
        row['scan_count_spearman_non_converter'] = r
        row['scan_count_veto'] = abs(r) > VETO_THRESHOLDS['scan_count_spearman'] if pd.notna(r) else None
    if 'cohort_probe_auc' in df.columns:
        cpa = df['cohort_probe_auc'].mean()
        row['cohort_probe_auc'] = cpa
        row['cohort_probe_escalation'] = cpa > 0.75 if pd.notna(cpa) else None
    demo_auc_by_cohort = {c: DEMO_FLOOR[c].mean() for c in DEMO_FLOOR.columns
                           if c.startswith('oof_auc_') and c != 'oof_auc'} if not DEMO_FLOOR.empty else {}
    for c, demo_auc in demo_auc_by_cohort.items():
        if c in df.columns:
            row[f'{c}_vs_demo_floor'] = df[c].mean() - demo_auc
            row[f'{c}_collapse_veto'] = df[c].mean() < demo_auc
    return row


VETO_TABLE = pd.DataFrame([veto_row(name, dirs) for name, dirs in RUNG_RUN_DIRS.items()]).set_index('rung')
VETO_TABLE


## Scan-count-shortcut mechanism (kept arms)

`common.visit_confound.within_subject_prob_slopes` needs a reloaded model + the `per_visit_probs` hook, not just the OOF frame -- run on the CV pool (never the test set) for a specific kept arm's best-fold checkpoint by setting `MECHANISM_CHECK_EXP_ID` below.

In [ ]:
MECHANISM_CHECK_EXP_ID = None  # e.g. 'tfgn-s1b-ssl-pooled-seed42' -- set to run this cell

if MECHANISM_CHECK_EXP_ID:
    from adapters import get_adapter
    from common.visit_confound import within_subject_prob_slopes
    from DATA.DELCODE.src.splitting.load_splits import splits_dir

    run_dir = resolve_source_run(MECHANISM_CHECK_EXP_ID, classifier_root=model_root)
    summary = load_summary(run_dir)
    if summary is None:
        raise FileNotFoundError(f'{MECHANISM_CHECK_EXP_ID}: no run_summary.json yet.')
    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}

    pooled_splits = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE' / 'SPLITS' / 'downstream'
    cv_pool_df = pd.concat([pd.read_csv(pooled_splits / 'train.csv'),
                            pd.read_csv(pooled_splits / 'val.csv')], ignore_index=True)

    adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
    adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift'}.get(adapter_key, adapter_key)
    adapter = get_adapter(adapter_key)(
        gaae_ckpt_path=summary.get('gaae_checkpoint') or '', gaae_hp=gaae_hp,
        train_config=summary['training_config'],
        data_root=str(repo_root / 'DATA/POOLED_ADNI_DELCODE/__fc_wholebrain_sch200_flat__/matrices'),
        cohorts_csv=None, device='cpu', rng=None,
    )
    state = adapter.load_state(run_dir)
    cv_bundle = adapter.prepare_data(cv_pool_df)
    slope_df, slope_stats = within_subject_prob_slopes(cv_bundle, adapter.per_visit_probs, state, device='cpu')
    print(f'Within-subject slope of P(converter) vs visit index -- {MECHANISM_CHECK_EXP_ID} (CV pool, not test):')
    for grp, s in slope_stats.items():
        print(f"  {grp:14s} median_slope={s['median_slope']:.4f}  frac_negative={s['frac_negative']}  n={s['n']}")
else:
    print('MECHANISM_CHECK_EXP_ID not set -- skipping (set it to a kept arm\'s seed id to run).')


## Gate-map validation (S2, S5) -- pre-registered, wired once those rungs exist

Permutation null, cross-fold Spearman stability, per-cohort split (`DOCS/temporal-first-ablation.md` "Gate-map validation") -- no-op until `gate_scores.npy` exists (S2 `use_gate: true` onward).

In [ ]:
GATE_RUNG = 'S2_gate'  # update once S2 is in RUNG_PREFIXES

if GATE_RUNG in RUNG_RUN_DIRS and RUNG_RUN_DIRS[GATE_RUNG]:
    gate_maps = []
    for exp_id, run_dir in RUNG_RUN_DIRS[GATE_RUNG].items():
        gp = run_dir / 'gate_scores.npy'
        if gp.is_file():
            gate_maps.append((exp_id, np.load(gp)))
    print(f'{len(gate_maps)} gate maps found for {GATE_RUNG}.')
    # Permutation null / cross-fold Spearman / per-cohort split go here once S2 exists
    # -- structure only, not computed on data that doesn't exist yet.
else:
    print(f'{GATE_RUNG}: no runs yet -- gate-map validation is a no-op until S2/S5 land.')


## Tier 4 — frozen reads (in-domain test + OASIS-3, exactly once)

**Gated.** Nothing below executes unless `RUN_FROZEN_READ = True` and `FROZEN_WINNER_ID` names the frozen winning arm's id prefix (no seed suffix -- all 4 seeds are read). Uses `common.frozen_read.score_frozen_split` -- reloads each seed's saved checkpoint, scores at its own OOF-derived threshold, records via the same `record_test_metrics` / `record_external_metrics` every non-deferred run already uses.

In [ ]:
if not RUN_FROZEN_READ:
    print('RUN_FROZEN_READ=False -- Tier 4 skipped (the ladder is not frozen yet, or '
          'this is a routine re-run of sections 1-6). Flip both flags above once ready.')
elif not FROZEN_WINNER_ID:
    raise ValueError('RUN_FROZEN_READ=True requires FROZEN_WINNER_ID (an id prefix).')
else:
    from common.frozen_read import score_frozen_split
    from DATA.DELCODE.src.splitting.load_splits import splits_dir

    gaae_hp_path = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
    gaae_hp = json.loads(gaae_hp_path.read_text()) if gaae_hp_path.is_file() else {}
    pooled_dir = repo_root / 'DATA' / 'POOLED_ADNI_DELCODE'
    in_domain_test_df = pd.read_csv(pooled_dir / 'SPLITS' / 'downstream' / 'test.csv')
    oasis_splits = repo_root / 'DATA' / 'OASIS3' / '__metadata__' / 'SPLITS' / 'downstream'
    oasis_test_df = pd.concat(
        [pd.read_csv(oasis_splits / f'{s}.csv') for s in ('train', 'val', 'test')], ignore_index=True,
    )
    oasis_test_df['cohort'] = 'oasis3'

    FROZEN_RESULTS = {}
    for seed in SEEDS:
        exp_id = f'{FROZEN_WINNER_ID}-seed{seed}'
        run_dir = resolve_source_run(exp_id, classifier_root=model_root)
        summary = load_summary(run_dir)
        if summary is None:
            raise FileNotFoundError(f'{exp_id}: no run_summary.json yet -- not ready for a frozen read.')
        adapter_key = str(summary.get('model_config', {}).get('model_type', '')).lower()
        adapter_key = {'tfgnclassifier': 'tfgn', 'logregdriftadapter': 'logregdrift'}.get(adapter_key, adapter_key)
        common_kwargs = dict(
            adapter_key=adapter_key,
            data_root=str(pooled_dir / '__fc_wholebrain_sch200_flat__' / 'matrices'),
            cohorts_csv=None, gaae_ckpt_path=summary.get('gaae_checkpoint') or '',
            gaae_hp=gaae_hp, device='cpu',
        )
        test_metrics = score_frozen_split(run_dir, in_domain_test_df, record_as='test', **common_kwargs)
        ext_metrics = score_frozen_split(run_dir, oasis_test_df, record_as='external', cohort='oasis3', **common_kwargs)
        FROZEN_RESULTS[exp_id] = {'test_auc': test_metrics['auc'], 'ext_oasis3_auc': ext_metrics['auc']}
        print(f'{exp_id}: test_auc={test_metrics["auc"]:.4f}  ext_oasis3_auc={ext_metrics["auc"]:.4f}')

    frozen_df = pd.DataFrame(FROZEN_RESULTS).T
    print()
    print('Frozen reads across seeds:')
    print(frozen_df)
    print()
    print(f"In-domain test AUC: {frozen_df['test_auc'].mean():.4f} +/- {frozen_df['test_auc'].std():.4f}")
    print(f"OASIS-3 AUC:        {frozen_df['ext_oasis3_auc'].mean():.4f} +/- {frozen_df['ext_oasis3_auc'].std():.4f}")

    winner_oof = RUNG_TABLES.get(FROZEN_WINNER_ID)
    if winner_oof is None:
        for name, prefix in RUNG_PREFIXES.items():
            if prefix == FROZEN_WINNER_ID:
                winner_oof = RUNG_TABLES.get(name)
    if winner_oof is not None and not winner_oof.empty:
        se_oof = winner_oof['oof_auc'].std(ddof=1) / np.sqrt(len(winner_oof))
        se_test = frozen_df['test_auc'].std(ddof=1) / np.sqrt(len(frozen_df)) if len(frozen_df) > 1 else float('nan')
        half_width = 1.96 * np.sqrt(se_oof ** 2 + se_test ** 2)
        lo, hi = winner_oof['oof_auc'].mean() - half_width, winner_oof['oof_auc'].mean() + half_width
        consistent = lo <= frozen_df['test_auc'].mean() <= hi
        print()
        print(f'Transport check: OOF={winner_oof["oof_auc"].mean():.4f}  '
              f'95% prediction interval=[{lo:.4f}, {hi:.4f}]  '
              f'test={frozen_df["test_auc"].mean():.4f}  '
              f'-> {"consistent" if consistent else "inconsistent"} with CV->test transport.')
        print('Winner\'s-curse statement: the OOF AUC above is expected to be optimistic '
              '(selected as the best of the ladder); the frozen test read above is the '
              'unbiased estimate. Report both, not the OOF number alone, as the headline.')


## Guard check — sections 1-6 touched no test/external metric

In [ ]:
_forbidden = {'TEST_METRICS', 'EXTERNAL_METRICS', 'test_df', 'in_domain_test_df', 'oasis_test_df'}
_touched = _forbidden & set(dir())
if RUN_FROZEN_READ:
    print(f'RUN_FROZEN_READ=True -- Tier 4 ran by design; test/external names present: {_touched or "(pandas frames only, as expected)"}.')
else:
    _unexpected = _touched - {'test_df'}  # 'test_df' would only exist if RUN_FROZEN_READ ran
    assert not _unexpected, f'Sections 1-6 touched test/external state unexpectedly: {_unexpected}'
    print('OK -- no test/external metric was read (RUN_FROZEN_READ=False).')
